import os
import torch

# Use all visible GPUs on Kaggle (usually already handled by PyTorch).
# This cell prints what is available so you can confirm resources.
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"CUDA available. GPUs visible: {gpu_count}")
    for i in range(gpu_count):
        print(f" - GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("CUDA is not available. Training will run on CPU.")

## 1) Setup Imports

In [ ]:
import os
import pickle
import random
import sys
from dataclasses import dataclass
from typing import Iterable, List, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

## 2) Repo Path + Model Imports

No fixed `PROJECT_ROOT` is used.
Set `REPO_PATH` to where the AttnGANEnhancement code exists in your Kaggle session.

In [ ]:
# If you cloned your repo in Kaggle, this is usually the right path:
REPO_PATH = "/kaggle/working/AttnGANEnhancement"

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

from config.kaggle_config import cfg
from src.bert_text_encoder import BERTTextEncoder
from src.model_wrapper import G_NET, RNN_ENCODER

print("REPO_PATH:", REPO_PATH)

## 3) Helper Functions

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_captions_pickle(pickle_path: str):
    with open(pickle_path, "rb") as fh:
        data = pickle.load(fh, encoding="latin1")
    train_caps, test_caps, ixtoword, wordtoix = data
    all_caps = list(train_caps) + list(test_caps)
    return all_caps, ixtoword, wordtoix


def prepare_caption_batch(captions: Sequence[Sequence[int]], max_words: int, device: torch.device):
    lengths = [min(len(c), max_words) for c in captions]
    max_len = max(max(lengths), 1)

    arr = np.zeros((len(captions), max_len), dtype=np.int64)
    for i, cap in enumerate(captions):
        trimmed = list(cap[:max_words])
        if trimmed:
            arr[i, : len(trimmed)] = np.asarray(trimmed, dtype=np.int64)

    cap_tensor = torch.LongTensor(arr).to(device)
    len_tensor = torch.LongTensor(lengths).to(device)
    return cap_tensor, len_tensor


def iterate_minibatches(items: Sequence[Sequence[int]], batch_size: int) -> Iterable[List[Sequence[int]]]:
    idxs = np.random.permutation(len(items))
    for start in range(0, len(idxs), batch_size):
        part = idxs[start : start + batch_size]
        yield [items[i] for i in part]

## 4) Training Config (Edit This Cell)

In [ ]:
# Fallback in case this cell runs before the repo-path cell.
if "REPO_PATH" not in globals():
    REPO_PATH = "/kaggle/working/AttnGANEnhancement"

# Kaggle dataset provided by you:
# https://www.kaggle.com/datasets/ahmednad/attngan-pretrained
DATA_DIR = "/kaggle/input/attngan-pretrained"
OUTPUT_DIR = "/kaggle/working/attngan_output"

@dataclass
class Args:
    captions_path: str = os.path.join(DATA_DIR, "captions.pickle")
    teacher_path: str = os.path.join(DATA_DIR, "text_encoder200.pth")
    generator_path: str = os.path.join(DATA_DIR, "bird_AttnGAN2.pth")
    output_dir: str = OUTPUT_DIR
    epochs: int = 5
    batch_size: int = 16
    lr: float = 2e-5
    word_loss_weight: float = 1.0
    seed: int = 123
    freeze_bert: bool = False


args = Args()
args

## 5) Train Function

In [ ]:
def train(args) -> None:
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg.CUDA = device.type == "cuda"
    cfg.TEXT.ENCODER_TYPE = "bert"

    out_dir = args.output_dir
    os.makedirs(out_dir, exist_ok=True)

    all_captions, ixtoword, wordtoix = load_captions_pickle(args.captions_path)
    print(f"[Train] Loaded {len(all_captions)} captions")

    teacher = RNN_ENCODER(len(wordtoix), nhidden=cfg.TEXT.EMBEDDING_DIM).to(device)
    teacher.load_state_dict(torch.load(args.teacher_path, map_location="cpu"))
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False

    student = BERTTextEncoder(
        ixtoword=ixtoword,
        embedding_dim=cfg.TEXT.EMBEDDING_DIM,
        model_name=cfg.TEXT.BERT_MODEL,
        max_words=cfg.TEXT.WORDS_NUM,
        freeze_bert=args.freeze_bert,
    ).to(device)
    student.train()

    netG = G_NET().to(device)
    netG.load_state_dict(torch.load(args.generator_path, map_location="cpu"))
    netG.eval()  # kept intact for compatibility

    optimizer = optim.AdamW(student.parameters(), lr=args.lr, weight_decay=1e-4)
    mse = nn.MSELoss()

    for epoch in range(1, args.epochs + 1):
        total_loss = 0.0
        total_steps = 0

        for batch_caps in iterate_minibatches(all_captions, args.batch_size):
            captions, cap_lens = prepare_caption_batch(batch_caps, cfg.TEXT.WORDS_NUM, device)

            with torch.no_grad():
                teacher_hidden = teacher.init_hidden(captions.size(0))
                teacher_words, teacher_sent = teacher(captions, cap_lens, teacher_hidden)

            student_words, student_sent = student(captions, cap_lens, None)

            max_t = min(student_words.size(2), teacher_words.size(2))
            sw = student_words[:, :, :max_t]
            tw = teacher_words[:, :, :max_t]

            token_mask = (~student.last_padding_mask[:, :max_t]).float().unsqueeze(1)
            word_loss = ((sw - tw) ** 2 * token_mask).sum() / token_mask.sum().clamp(min=1.0)
            sent_loss = mse(student_sent, teacher_sent)
            loss = sent_loss + args.word_loss_weight * word_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 5.0)
            optimizer.step()

            total_loss += float(loss.item())
            total_steps += 1

        avg_loss = total_loss / max(total_steps, 1)
        print(f"[Train] Epoch {epoch:02d}/{args.epochs} - loss: {avg_loss:.6f}")

    encoder_out = os.path.join(out_dir, "bert_text_encoder.pth")
    generator_out = os.path.join(out_dir, "bert_AttnGAN_generator.pth")

    torch.save(student.state_dict(), encoder_out)
    torch.save(netG.state_dict(), generator_out)

    print("\n[Train] Saved:")
    print(f"  - {encoder_out}")
    print(f"  - {generator_out}")
    print("\n[Kaggle] Load with:")
    print('  MODEL_DIR = "/kaggle/input/your-model-folder"')
    print('  cfg.TRAIN.NET_E = "bert_text_encoder.pth"')
    print('  cfg.TRAIN.NET_G = "bert_AttnGAN_generator.pth"')
    print('  cfg.TEXT.ENCODER_TYPE = "bert"')

## 6) Run Training

In [ ]:
required_files = [args.captions_path, args.teacher_path, args.generator_path]
print("Checking required files...")
for p in required_files:
    print(f"{p} ->", "OK" if os.path.exists(p) else "MISSING")

os.makedirs(args.output_dir, exist_ok=True)
print("Output dir:", args.output_dir)

In [ ]:
train(args)